In [268]:
import pandas as pd
import numpy as np

In [269]:
dataset = pd.read_csv(r'C:\Projects_ciência_dados\students_performance\data\interim\dataset_interim.csv')

In [270]:
dataset

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,Desempenho
0,female,group B,bachelor's degree,standard,none,72,72,74,1
1,female,group C,some college,standard,completed,69,90,88,1
2,female,group B,master's degree,standard,none,90,95,93,1
3,male,group A,associate's degree,free/reduced,none,47,57,44,0
4,male,group C,some college,standard,none,76,78,75,1
...,...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95,1
996,male,group C,high school,free/reduced,none,62,55,55,0
997,female,group C,high school,free/reduced,completed,59,71,65,0
998,female,group D,some college,standard,completed,68,78,77,1


##### Eliminar notas das provas que foram usadas para fazer a classe, senão será um data leakage

###### Deixar só mais math score é a melhor decisão, pois o modelo fica menos dependente

In [ ]:
dataset = dataset.drop(["writing score", "reading score"], axis=1) # Deixar apenas math pq tem menos correlação com as outras

##### Verificar nulos

In [272]:
dataset.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
Desempenho                     0
dtype: int64

##### Verificar valores inconsistentes

In [273]:
negativas = dataset.select_dtypes(include=["int", "float"])
(negativas<0).any().sum()

np.int64(0)

##### Separar em variáveis previsoras e classe

In [274]:
x_previsoras = dataset.drop("Desempenho", axis=1)

In [275]:
x_previsoras

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score
0,female,group B,bachelor's degree,standard,none,72
1,female,group C,some college,standard,completed,69
2,female,group B,master's degree,standard,none,90
3,male,group A,associate's degree,free/reduced,none,47
4,male,group C,some college,standard,none,76
...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88
996,male,group C,high school,free/reduced,none,62
997,female,group C,high school,free/reduced,completed,59
998,female,group D,some college,standard,completed,68


In [276]:
y_classe = dataset["Desempenho"]

In [277]:
y_classe

0      1
1      1
2      1
3      0
4      1
      ..
995    1
996    0
997    0
998    1
999    1
Name: Desempenho, Length: 1000, dtype: int64

##### Fazer split

In [278]:
from sklearn.model_selection import train_test_split

In [279]:
x_train, x_test, y_train, y_test = train_test_split(x_previsoras, y_classe, test_size=0.20, random_state=42)

In [280]:
x_train.shape

(800, 6)

In [281]:
x_test.shape

(200, 6)

In [282]:
y_train.shape

(800,)

In [283]:
y_test.shape

(200,)

#### Encontrar variáveis categóricas

In [ ]:
categoricas = x_train.select_dtypes(include=["object"]).columns.to_list()

In [285]:
x_train.columns

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course', 'math score'],
      dtype='object')

In [ ]:
categoricas

##### Encontrar variáveis numéricas

In [ ]:
numericas = x_train.select_dtypes(include=["int", "float"]).columns.to_list()
numericas

Aplicar OneHotEncoder nas categóricas e StandarScaler nas numéricas

In [287]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [288]:
preprocessor = ColumnTransformer([
    ('one_hot', OneHotEncoder(sparse_output=False), categoricas),
    ('scaler', StandardScaler(), numericas)
])

In [ ]:
preprocessor.set_output(transform="pandas")

,transformers,"[('one_hot', ...), ('scaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [290]:
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [291]:
x_train

,one_hot__gender_female,one_hot__gender_male,one_hot__race/ethnicity_group A,one_hot__race/ethnicity_group B,one_hot__race/ethnicity_group C,one_hot__race/ethnicity_group D,one_hot__race/ethnicity_group E,one_hot__parental level of education_associate's degree,one_hot__parental level of education_bachelor's degree,one_hot__parental level of education_high school,one_hot__parental level of education_master's degree,one_hot__parental level of education_some college,one_hot__parental level of education_some high school,one_hot__lunch_free/reduced,one_hot__lunch_standard,one_hot__test preparation course_completed,one_hot__test preparation course_none,scaler__math score
29,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.299452
535,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,-0.033050
695,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.832756
557,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.366053
836,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.433153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.365559
270,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.166751
860,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.898857
435,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,-1.098658


In [292]:
x_train.to_csv('../data/processed/x_train.csv', index=False)

In [293]:
x_test.to_csv('../data/processed/x_test.csv', index=False)

In [294]:
y_train.to_csv('../data/processed/y_train.csv', index=False)

In [295]:
y_test.to_csv('../data/processed/y_test.csv', index=False)